**Eye-Tracking & Neurocognitive Intelligence | Eye-Tracking Foundations & Signal Analysis | Presented by: Reza Saadatyar (2026)**

**05 | Participant-Balanced Descriptive Statistics**

![Observed participant-level task statistics and exploratory bootstrap intervals](../../images/05_participant_balanced_statistics.svg)

**Objective:** Describe gaze quality and detector-derived gaze features across the four GazeBase tasks while treating the **participant**, rather than an eye sample, event or recording, as the independent resampling unit.

**Observed dataset:** GazeBase v2.0, Round 1, Session 1. Notebook 05 reconciled **40 per-record feature rows from 10 participants** with Notebook 01's original record-quality table. Every record identity, participant and task matched; the maximum absolute difference in saved valid gaze fraction was **0.0**.

**Executed analysis:** The 40 participant-by-task rows were summarised using four prespecified metrics: valid gaze fraction, mean I-VT fixation duration, approximate I-VT fixation-centroid path length and mean I-VT saccade peak speed. The saved statistics contain **16 task-by-feature summaries**, each supported by **10 independent participants**. Exploratory percentile bootstrap intervals used **500 participant-level resamples, seed 17**.

**Recorded gaze validity means:** FXS **97.04%** (95.97–97.97%), HSS **94.22%** (86.39–99.25%), RAN **97.48%** (94.74–99.22%) and TEX **94.04%** (88.96–97.98%). Ranges are exploratory 2.5th–97.5th bootstrap percentiles, **not** confirmatory task-comparison tests.

**Scientific boundaries:** Ten people are not a representative population. Derived durations, scanpath lengths and velocities depend on provisional I-VT definitions; extreme calculated speeds from Notebook 02 remain unresolved. No causality, reading comprehension, independent generalisation or statistical significance is inferred.


**00. Colab Repository Setup**

In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


/content/Eye-Tracking-Neurocognitive-Intelligence


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eye_tracking_foundations")

# ----------------- 2. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 3. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

# ------------------------ 4. Study Research Utilities ------------------------
from core.stage_outputs import ensure_stage_inputs
from gaze_source import read_stage
from gaze_statistics import (
    validate_record_quality_alignment, participant_summary,
    summarise_participant_metrics,
)

# --------------------- 5. Output and Presentation Setup ----------------------
output_dir = root / "eye_tracking_foundations" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
plot_style(
    font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5,
    label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(10, 4),
)
print("Current outputs:", output_dir)


Repository up to date.
Current outputs: /content/Eye-Tracking-Neurocognitive-Intelligence/eye_tracking_foundations/output


**02. Recover and Reconcile Notebook 01 and Notebook 04 Tables**

Use the compact `01_quality_by_record.csv` and `04_features_by_record.csv` tables, without loading multi-million-row gaze recordings. Each supplies `record_id`, `participant_id`, `task` and `valid_fraction`.

The executed identity check matched all **40 records** one-to-one, represented all **10 participants** and the FXS/HSS/RAN/TEX tasks, and found **zero maximum difference** between the original-audit and feature-table valid gaze fractions. This establishes consistent recorded inputs for the descriptive analysis; it does not independently verify the underlying eye tracker or detector labels. Any discrepancy in future runs raises an error rather than silently dropping or replacing observations.


In [ ]:
# =============== 02. Recover Feature and Source Quality Tables ===============

# ------------------------ 1. Verify Notebook Outputs -------------------------
ensure_stage_inputs(output_dir, 1, ("01_quality_by_record.csv",))
ensure_stage_inputs(output_dir, 4, ("04_features_by_record.csv",))
quality = read_stage(
    output_dir / "01_quality_by_record.csv",
    ("record_id", "participant_id", "task", "valid_fraction"),
)
features = read_stage(
    output_dir / "04_features_by_record.csv",
    ("record_id", "participant_id", "task", "valid_fraction"),
)

# ----------------- 2. Reconcile Exact Source Record Identity -----------------
quality_alignment = validate_record_quality_alignment(features, quality)
print("Matching source records:", len(quality_alignment))
print("Independent participants:", features["participant_id"].nunique())
print("Tasks:", ", ".join(sorted(features["task"].unique())))
print("Maximum validity discrepancy:",
      quality_alignment["absolute_valid_fraction_difference"].max())
display(quality_alignment[[
    "record_id", "participant_id_notebook04", "task_notebook04",
    "valid_fraction_notebook04", "valid_fraction_notebook01",
]].head(8))


Matching source records: 40
Independent participants: 10
Tasks: FXS, HSS, RAN, TEX
Maximum validity discrepancy: 0.0


,record_id,participant_id_notebook04,task_notebook04,valid_fraction_notebook04,valid_fraction_notebook01
0,S_1001_S1_FXS,001,FXS,0.967566,0.967566
1,S_1001_S1_HSS,001,HSS,0.996387,0.996387
2,S_1001_S1_RAN,001,RAN,0.986779,0.986779
3,S_1001_S1_TEX,001,TEX,0.966713,0.966713
4,S_1002_S1_FXS,002,FXS,0.968356,0.968356
5,S_1002_S1_HSS,002,HSS,0.990959,0.990959
6,S_1002_S1_RAN,002,RAN,0.990869,0.990869
7,S_1002_S1_TEX,002,TEX,0.985557,0.985557


**03. Aggregate Within Participants and Tasks**

Repeated source recordings are not independent people. For each feature, aggregate eligible records within the same `participant_id` and `task` before calculating task-level statistics. The current GazeBase selection contains one recording of each of the four tasks for each of ten people, yielding **40 participant-by-task rows**.

The four prespecified metrics are `valid_fraction`, `mean_fixation_ms`, `scanpath_length_dva` and `mean_saccade_peak_dva_s`. These represent observed recording quality or quantities derived from provisional I-VT events, not cognitive outcomes. In particular, the fixation-centroid path is an approximation rather than a stimulus-aligned reading trajectory, and the high computed velocities noted in Notebook 02 remain relevant to speed summaries.

The executed summary shows ten unique `participant_id` values. Group-level means do not inflate the sample size using gaze samples or event intervals.


In [4]:
# ===================== 03. Aggregate Within Participants =====================

# ---------------------- 1. Prespecified Source Metrics -----------------------
METRICS = (
    "valid_fraction", "mean_fixation_ms", "scanpath_length_dva",
    "mean_saccade_peak_dva_s",
)
participant = participant_summary(features, METRICS)
participant.to_csv(output_dir / "05_participant_features.csv", index=False)

# -------------------- 2. Show Independent Analysis Units ---------------------
print("Participant-by-task rows:", len(participant))
print("Unique participants:", participant["participant_id"].nunique())
display(participant.head(12))


Participant-by-task rows: 40
Unique participants: 10


,task,participant_id,valid_fraction,mean_fixation_ms,scanpath_length_dva,mean_saccade_peak_dva_s
0,FXS,001,0.967566,387.166667,8.706277,247.562896
1,FXS,002,0.968356,141.000000,8.415860,394.341395
2,FXS,003,0.986599,1813.000000,9.901850,297.212592
3,FXS,004,0.963606,643.818182,20.694977,107.010645
4,FXS,005,0.932639,188.551020,240.408846,577.363500
5,FXS,006,0.978715,106.576923,5.386057,1468.296745
6,FXS,007,0.974857,141.048780,65.044052,638.067700
7,FXS,008,0.958665,1096.153846,20.115494,106.523974
8,FXS,009,0.974526,535.750000,10.450404,2811.997389
9,FXS,010,0.998674,407.542857,9.337436,276.172157


**04. Calculate Exploratory Participant-Level Uncertainty**

For each task and metric, calculate the mean, sample standard deviation and a percentile interval from **500 bootstrap resamples of participants**, with replacement and fixed random seed **17**. A participant contributes one aggregated value per task. Missing feature values, if encountered, are excluded per metric and the actual number of independent contributors is reported. When fewer than two people contribute, interval bounds are undefined.

The executed `05_descriptive_statistics.csv` contains **16 rows** (four tasks × four metrics), all with **10 independent participants**. The interval bounds are exploratory 2.5th and 97.5th percentiles, not guarantees of population coverage or formal between-task significance.

The same ten participants completed all four tasks, but this Notebook does **not** calculate paired task-difference intervals or infer task effects. The particularly wide variation in some detector-derived measures and the limited independent cohort should be retained in interpretation.


In [ ]:
# ================ 04. Calculate Participant-Level Uncertainty ================

# ---------------------- 1. Fixed Exploratory Bootstrap -----------------------
BOOTSTRAP_DRAWS = 500
BOOTSTRAP_SEED = 17
stats = summarise_participant_metrics(
    participant, METRICS, draws=BOOTSTRAP_DRAWS, seed=BOOTSTRAP_SEED
)
stats.to_csv(output_dir / "05_descriptive_statistics.csv", index=False)

# ---------------------- 2. Check Supported Feature Rows ----------------------
print("Descriptive task-feature rows:", len(stats))
print("Unique source tasks:", stats["task"].nunique())
display(stats[[
    "task", "feature", "n_independent_participants", "participant_mean",
    "participant_sd", "bootstrap_ci_lower", "bootstrap_ci_upper",
    "interval_status",
]])


Descriptive task-feature rows: 16
Unique source tasks: 4


,task,feature,n_independent_participants,participant_mean,participant_sd,bootstrap_ci_lower,bootstrap_ci_upper,interval_status
0,FXS,valid_fraction,10,0.970420,0.017606,0.959669,0.979730,exploratory_cluster_bootstrap
1,FXS,mean_fixation_ms,10,546.060828,538.609629,269.836265,898.929086,exploratory_cluster_bootstrap
2,FXS,scanpath_length_dva,10,39.846125,72.612366,10.391052,84.379376,exploratory_cluster_bootstrap
3,FXS,mean_saccade_peak_dva_s,10,692.454899,844.127644,285.841779,1208.893045,exploratory_cluster_bootstrap
4,HSS,valid_fraction,10,0.942191,0.108453,0.863920,0.992464,exploratory_cluster_bootstrap
5,HSS,mean_fixation_ms,10,244.955480,97.466600,191.039937,313.362038,exploratory_cluster_bootstrap
6,HSS,scanpath_length_dva,10,2366.785279,986.812923,1722.151990,2870.765983,exploratory_cluster_bootstrap
7,HSS,mean_saccade_peak_dva_s,10,408.580379,96.887505,353.303856,467.294019,exploratory_cluster_bootstrap
8,RAN,valid_fraction,10,0.974758,0.038034,0.947405,0.992207,exploratory_cluster_bootstrap
9,RAN,mean_fixation_ms,10,253.272044,129.252916,177.464827,332.261482,exploratory_cluster_bootstrap


**05. Plot Participant-Weighted Descriptive Findings**

Plot mean `valid_fraction` by task and its observed exploratory 2.5th–97.5th bootstrap percentile interval, with `n=10` displayed for each task. The recorded means are FXS **0.9704**, HSS **0.9422**, RAN **0.9748** and TEX **0.9404**. All four intervals were obtained by resampling participant-level values, not millions of eye samples.

For legibility, the chart uses a **clearly labelled zoomed vertical axis (0.82–1.02)** rather than 0–1; this enlarges the interval region and must not be mistaken for a proportionate zero-based comparison. Numerical estimates in `05_descriptive_statistics.csv` remain the authoritative results.

The overlapping exploratory intervals do not themselves establish equivalence or non-significance. Task differences can reflect stimulus protocol, participant variation, validity and calibration conditions; no causal or cognitive outcome is claimed.


In [ ]:
# ============ 05. Plot Participant-Weighted Descriptive Findings =============

# ---------------- 1. Select Supported Gaze-Quality Estimates -----------------
valid = (
    stats.loc[stats["feature"].eq("valid_fraction")]
    .sort_values("task").reset_index(drop=True)
)
if valid.empty:
    raise ValueError("No participant-level valid-fraction summary to plot.")

# ------------------- 2. Draw Means and Bootstrap Intervals -------------------
fig, ax = plt.subplots(figsize=(8.5, 4.5))
positions = np.arange(len(valid))
ax.scatter(
    positions, valid["participant_mean"], marker="o", s=45,
    label="Participant mean", zorder=3,
)
for i, row in valid.iterrows():
    lo, hi = row["bootstrap_ci_lower"], row["bootstrap_ci_upper"]
    if np.isfinite(lo) and np.isfinite(hi):
        ax.vlines(
            i, lo, hi, lw=2.1,
            label="Bootstrap percentiles (2.5–97.5%)" if i == 0 else None,
        )
        ax.hlines([lo, hi], i - 0.09, i + 0.09, lw=1.6)
    ax.annotate(
        f"n={int(row['n_independent_participants'])}",
        (i, float(row["participant_mean"])),
        xytext=(0, 9), textcoords="offset points",
        ha="center", fontsize=9,
    )
ax.set_xticks(positions, valid["task"])
ax.set(
    ylabel="Participant-level valid gaze fraction",
    xlabel="Gaze task",
    ylim=(0.82, 1.02),
    title="Exploratory Gaze Quality by Task (Zoomed Axis)",
)
ax.set_xlim(-0.55, len(valid) - 0.45)
ax.grid(axis="y", alpha=0.2)
ax.text(
    0.99, 0.03, "Zoomed y-axis: 0.82–1.02",
    transform=ax.transAxes, ha="right", va="bottom", fontsize=8,
)
ax.legend(loc="lower right")
fig.tight_layout()
save_figure(fig, output_dir / "05_statistics.png")
plt.show()


**06. Scientific Interpretation and Output Download**

The executed Notebook 05 produces three named outputs:

- `05_participant_features.csv`: 40 participant-by-task records built from the aligned source feature table.
- `05_descriptive_statistics.csv`: 16 task-by-metric rows containing participant support, mean, standard deviation and exploratory percentile limits.
- `05_statistics.png`: the task-wise gaze validity plot with clearly labelled participant-bootstrap ranges.

Every metric has support from ten people in this selected subset, but that support remains **exploratory**. The observed means and intervals do not validate an eye-movement detector, justify selecting a threshold, prove reading or cognitive differences, or support population generalisation.

Notebook 06 is a **separate sensitivity analysis** of the predefined I-VT threshold grid. It uses Notebook 02's original prepared motion data and existing vendor annotations, not the bootstrap intervals from this notebook. Keep generated CSVs, plots and ZIPs private unless explicitly selected for publication.


In [ ]:
# ================ 06. Scientific Interpretation and Download =================

# ------------------- 1. Confirm Current Scientific Outputs -------------------
print("Participant summaries:", output_dir / "05_participant_features.csv")
print("Task statistics:", output_dir / "05_descriptive_statistics.csv")
print("Uncertainty figure:", output_dir / "05_statistics.png")
print("Independent unit: participant; intervals are exploratory.")

# ----------------------- 2. Download Notebook Outputs ------------------------
download_outputs_from_colab("eye_tracking_foundations/output", "05")
